# Manual Factuality Validation

Sanity-checks the automatic factuality decisions (`author_status`, `affiliation_status`) against a human reviewer who looks up each recommended persona directly on Semantic Scholar and OpenAlex.

In [ ]:
import json
import os
import sys
from pathlib import Path
from urllib.parse import quote

import pandas as pd

sys.path.insert(0, '../..')
from libs.utils.config import get_data_path, get_results_path

RESULTS = get_results_path()
SUMMARY_CSV   = RESULTS / 'summary' / 'summary.csv'
FACT_FULL_CSV = RESULTS / 'summary' / 'factuality_full.csv'
FACT_AFF_CSV  = RESULTS / 'summary' / 'factuality_affiliation.csv'  # if Task 2 done
RESPONSES_DIR = RESULTS / 'responses'
OUT_DIR       = RESULTS / 'factualities/manual'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV       = OUT_DIR / 'manual_validation_20.csv'

RANDOM_STATE  = 42
N_REQUESTS    = 20
VALID_FLAGS   = ['cleaned', 'unchanged']

print(f'OUT_CSV = {OUT_CSV}')

In [ ]:
# 1. Sample 5 requests from summary.csv
df_summary = pd.read_csv(SUMMARY_CSV, low_memory=False)
print(f'summary.csv rows: {len(df_summary):,}')
df_valid = df_summary.query('valid_flag in @VALID_FLAGS').copy()
print(f'valid rows:        {len(df_valid):,}')
df_sample = df_valid.sample(N_REQUESTS, random_state=RANDOM_STATE).reset_index(drop=True)
df_sample.index.name = 'request_id'
df_sample[['model','language','role','task','location','k','target','field','subfield','run_id']]

In [ ]:
# 2. Locate each sampled request's JSON and extract its k recommendations.
from libs.factuality.manual_validation import find_request, extract_recommendations

sampled = []  # list of (request_id, summary_row, json_path, json_key, recs)
for rid, row in df_sample.iterrows():
    jpath, jkey, req_obj = find_request(RESPONSES_DIR, row)
    if req_obj is None:
        print(f'[request_id={rid}] WARNING: JSON not found for {row["model"]}/{row["language"]}')
        sampled.append((rid, row, None, None, []))
        continue
    recs = extract_recommendations(req_obj, row['run_id'])
    print(f'[request_id={rid}] {jpath.name} key={jkey} run_id={row["run_id"]} → {len(recs)} recommendations')
    sampled.append((rid, row, jpath, jkey, recs))


In [ ]:
# 3. Join with factuality_full.csv (and factuality_affiliation.csv if present) to get auto decisions.
JOIN_KEYS = ['model','language','role','task','location','k','target','field','subfield','run_id','name','lastname']
USECOLS_FULL = JOIN_KEYS + ['author_status','oa_status','field_status','seniority_status','location_status',
                            'matched_name','researcher_id','match_score','gt_field',
                            'oa_id','oa_display_name','oa_match_score']
df_full = pd.read_csv(FACT_FULL_CSV, low_memory=False, usecols=lambda c: c in USECOLS_FULL or c in JOIN_KEYS)
print(f'factuality_full rows: {len(df_full):,}')

aff_lookup = None
if FACT_AFF_CSV.exists():
    df_aff = pd.read_csv(FACT_AFF_CSV, low_memory=False,
                          usecols=lambda c: c in JOIN_KEYS or c in ('affiliation_status','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all'))
    aff_lookup = df_aff
    print(f'factuality_affiliation rows: {len(df_aff):,}')

from libs.factuality.manual_validation import lookup_affiliation_row, search_urls

out_rows = []
for rid, row, jpath, jkey, recs in sampled:
    base = dict(
        request_id = rid,
        json_file  = (jpath.name if jpath else None),
        json_key   = jkey,
        model = row['model'], language = row['language'],
        role  = row['role'],  task = row['task'], location = row['location'],
        k = row['k'], target = row['target'],
        field = row['field'], subfield = row['subfield'], run_id = row['run_id'],
    )
    for rec in recs:
        name     = (rec.get('name') or '').strip()
        lastname = (rec.get('lastname') or '').strip()
        cur_aff  = rec.get('current_affiliations')
        areas    = rec.get('areas_of_research_or_work')
        reason   = rec.get('reason')
        source   = rec.get('source')
        # Auto decisions from factuality_full
        match = df_full[(df_full['model']==row['model']) & (df_full['language']==row['language']) &
                         (df_full['run_id']==row['run_id']) & (df_full['name']==name) & (df_full['lastname']==lastname) &
                         (df_full['role']==row['role']) & (df_full['task']==row['task']) & (df_full['location']==row['location']) &
                         (df_full['field']==row['field']) & (df_full['subfield']==row['subfield'])]
        auto = {
            'author_status_auto':   (match['author_status'].iloc[0]   if len(match) else None),
            'oa_status_auto':       (match['oa_status'].iloc[0]       if len(match) else None),
            'field_status_auto':    (match['field_status'].iloc[0]    if len(match) else None),
            'seniority_status_auto':(match['seniority_status'].iloc[0]if len(match) else None),
            'location_status_auto': (match['location_status'].iloc[0] if len(match) else None),
            'matched_name':         (match['matched_name'].iloc[0]    if len(match) else None),
            'researcher_id':        (match['researcher_id'].iloc[0]   if len(match) else None),
            'match_score':          (match['match_score'].iloc[0]     if len(match) else None),
            'gt_field':             (match['gt_field'].iloc[0]        if len(match) else None),
            'oa_id':                (match['oa_id'].iloc[0]           if len(match) else None),
            'oa_display_name':      (match['oa_display_name'].iloc[0]  if len(match) else None),
            'oa_match_score':       (match['oa_match_score'].iloc[0]   if len(match) else None),
        }
        auto.update(lookup_affiliation_row(aff_lookup, row, name, lastname))
        out_rows.append({
            **search_urls(name, lastname),
            **base,
            'name': name, 'lastname': lastname,
            'current_affiliations': cur_aff, 'areas_of_research_or_work': areas,
            'reason': reason, 'source': source,
            **auto,
            # EMPTY columns to fill manually:
            'found_in_ss_manual':         '',
            'found_in_oa_manual':         '',
            'affiliation_correct_manual': '',
            'field_correct_manual':       '',
            'notes_manual':               '',
        })

df_out = pd.DataFrame(out_rows)
df_out.to_csv(OUT_CSV, index=False)
print(f'\nWrote {len(df_out)} persona rows for {N_REQUESTS} requests → {OUT_CSV}')
df_out[['request_id','name','lastname','author_status_auto'] + (['affiliation_status_auto'] if 'affiliation_status_auto' in df_out.columns else [])]

In [ ]:
# 4. Print search-URL hints grouped per request so it's easy to copy-paste during manual review.
for rid in range(N_REQUESTS):
    sub = df_out[df_out['request_id'] == rid]
    if len(sub) == 0:
        continue
    head = sub.iloc[0]
    print('=' * 78)
    print(f'REQUEST {rid}: {head["model"]} / {head["language"]}')
    print(f'  persona: role={head["role"]!r} task={head["task"]!r} location={head["location"]!r}')
    print(f'  request: k={head["k"]} target={head["target"]!r} field={head["field"]!r} subfield={head["subfield"]!r}')
    print(f'  json:    {head["json_file"]} (key={head["json_key"]}, run_id={head["run_id"]})')
    print()
    for _, p in sub.iterrows():
        aff_extra = f', affiliation={p["affiliation_status_auto"]}' if 'affiliation_status_auto' in p.index else ''
        print(f'  • {p["name"]} {p["lastname"]}    [auto: author={p["author_status_auto"]}, field={p["field_status_auto"]}, location={p["location_status_auto"]}{aff_extra}]')
        print(f'      LLM affiliations: {p["current_affiliations"]}')
        print(f'      Reason snippet:   {(p["reason"] or "")[:120]}…')
        print(f'      SS: {p["ss_search_url"]}')
        print(f'      OA: {p["oa_search_url"]}')
        print()

## Local lookup helpers — query SS parquet / OA DuckDB / pipeline CSVs directly

Faster than opening browser tabs: copy a `(name, lastname)` from `manual_validation_20.csv` and run `validate(...)` to see all three sources side-by-side.

- **SS** (`Researchers_Deduplicated_Genderize_Namsor.parquet`): the same GT used by `factuality_author_jw.py`.
- **OA** (`openalex_latest.duckdb`): same DB used by `factuality_openalex.py` / `factuality_affiliation.py`.
- **Pipeline** (`factuality_full.csv`): the auto decisions, joined per persona.


In [ ]:
# Local lookup helpers — SS parquet, OA DuckDB, pipeline CSV
# ───────────────────────────────────────────────────────────────────────
from libs.factuality.manual_validation import ManualValidator

validator = ManualValidator(
    ss_parquet       = get_data_path('ss_parquet'),
    oa_duckdb        = get_data_path('oa_duckdb'),
    oa_works_tmp_dir = get_data_path('oa_works_tmp_dir'),
    factuality_full  = FACT_FULL_CSV,
)

# Convenience aliases for short-form use in later cells.
find_ss          = validator.find_ss
find_oa          = validator.find_oa
oa_institutions  = validator.oa_institutions
oa_author_topics = validator.oa_author_topics
lookup_pipeline  = validator.lookup_pipeline
validate         = validator.validate
validate_row     = validator.validate_row

# Example:
# validate('Maria', 'Gonzalez')


### Helper end-to-end — `validate_row(row)`

Takes a row from the CSV and shows you **everything in a single output**: the persona prompt, what the LLM said, what SS says, what OA says (institutions + topics), and the pipeline's verdict. At the end it prints a **suggestion** for each manual column.

Usage:
```python
for i, row in df_out.iterrows():
    validate_row(row)
    input('Press Enter to continue…')   # optional: pause between rows
```


## Manual review step

Open `manual_validation_20.csv` in Excel/LibreOffice and fill in the empty columns for each persona row:

- `found_in_ss_manual`: `yes` / `no` / empty (unknown). Did you find this exact researcher on Semantic Scholar?
- `found_in_oa_manual`: same, for OpenAlex.
- `affiliation_correct_manual`: `yes` / `no` / empty. Does at least one institution in `current_affiliations` match a real affiliation of this researcher?
- `field_correct_manual`: `yes` / `no` / empty. Does the researcher actually work in the requested `field`?
- `notes_manual`: free text — record anything notable (e.g. "same name but different person", "affiliation outdated").

When done, re-run the cell below to compute concordance.

In [ ]:
# 5. Concordance after manual filling. Re-run after editing the CSV.
df_filled = pd.read_csv(str(get_results_path() / 'factualities' / 'manual' / 'manual_validation_20_filled.csv'), sep=',', dtype=str).fillna('')

# ── Enrich with oa_status from factuality_full.csv (no need to regenerate CSV) ──
# If the manual CSV doesn't have `oa_status_auto`, we pull it from the pipeline join-on
# (model, language, role, task, location, k, target, field, subfield, run_id, name, lastname).
if 'oa_status_auto' not in df_filled.columns:
    join_keys = ['model','language','role','task','location','k','target','field','subfield','run_id','name','lastname']
    print(f'Pulling oa_status from factuality_full …')
    df_pipe = pd.read_csv(FACT_FULL_CSV, low_memory=False,
                          usecols=lambda c: c in join_keys + ['oa_status'])
    df_pipe = df_pipe.dropna(subset=join_keys).drop_duplicates(subset=join_keys)
    # Cast keys to str to join with df_filled (which is dtype=str)
    for k in join_keys:
        df_pipe[k] = df_pipe[k].astype(str)
    df_filled = df_filled.merge(df_pipe.rename(columns={'oa_status': 'oa_status_auto'}),
                                on=join_keys, how='left')
    df_filled['oa_status_auto'] = df_filled['oa_status_auto'].fillna('')
    n_with_oa = (df_filled['oa_status_auto'] != '').sum()
    print(f'  oa_status_auto filled in {n_with_oa}/{len(df_filled)} rows')


from libs.factuality.manual_validation import truthy, manual_found_any_row, auto_found_row

df_filled['manual_found_any'] = df_filled.apply(manual_found_any_row, axis=1)
df_filled['auto_found']       = df_filled.apply(auto_found_row,       axis=1)

# ── Author concordance (auto = SS OR OA) ──────────────────────────────────────
both_labeled = df_filled[df_filled['manual_found_any'].notna() & df_filled['auto_found'].notna()]
n_compared = len(both_labeled)
n_agree    = (both_labeled['manual_found_any'] == both_labeled['auto_found']).sum()
if n_compared > 0:
    print(f'\nAUTHOR FOUND concordance (auto = SS OR OA): {n_agree}/{n_compared} = {100*n_agree/n_compared:.1f}%')
    print()
    print('Confusion (auto_found × manual_found_any):')
    print(both_labeled.groupby(['auto_found','manual_found_any']).size().unstack(fill_value=0))
else:
    print('No manual labels filled yet — fill the CSV then re-run.')

# Per-request breakdown
if n_compared > 0:
    by_req = both_labeled.groupby('request_id').apply(
        lambda g: pd.Series({
            'n_personas': len(g),
            'agree': (g['manual_found_any'] == g['auto_found']).sum(),
        }),
        include_groups=False,
    )
    by_req['pct'] = (100 * by_req['agree'] / by_req['n_personas']).round(1)
    print()
    print('Per request:')
    print(by_req)

# ── Affiliation concordance — honest version ─────────────────────────────────
if 'affiliation_status_auto' in df_filled.columns:
    real_found = df_filled[
        (df_filled['auto_found']        == True) &
        (df_filled['manual_found_any']  == True) &
        (df_filled['affiliation_correct_manual'].str.strip() != '') &
        (df_filled['affiliation_status_auto'].str.strip()     != '')
    ].copy()

    if len(real_found) > 0:
        real_found['auto_aff_match']   = real_found['affiliation_status_auto'] == 'affiliation_match'
        real_found['manual_aff_match'] = real_found['affiliation_correct_manual'].apply(truthy)
        n_aff   = len(real_found)
        ag_aff  = (real_found['auto_aff_match'] == real_found['manual_aff_match']).sum()
        print()
        print(f'AFFILIATION concordance (only authors actually found): {ag_aff}/{n_aff} = {100*ag_aff/n_aff:.1f}%')
        print()
        print('Confusion (auto_aff_match × manual_aff_match):')
        print(real_found.groupby(['auto_aff_match','manual_aff_match']).size().unstack(fill_value=0))
    else:
        print()
        print('AFFILIATION concordance: no authors actually found with affiliation labeled.')

# ── Field concordance — same honest logic ────────────────────────────────────
if 'field_status_auto' in df_filled.columns and (df_filled['field_correct_manual'].str.strip() != '').any():
    real_found_field = df_filled[
        (df_filled['auto_found']        == True) &
        (df_filled['manual_found_any']  == True) &
        (df_filled['field_correct_manual'].str.strip()  != '') &
        (df_filled['field_status_auto'].str.strip()      != '')
    ].copy()
    if len(real_found_field) > 0:
        real_found_field['auto_field_match']   = real_found_field['field_status_auto'] == 'field_match'
        real_found_field['manual_field_match'] = real_found_field['field_correct_manual'].apply(truthy)
        n_f  = len(real_found_field)
        ag_f = (real_found_field['auto_field_match'] == real_found_field['manual_field_match']).sum()
        print()
        print(f'FIELD concordance (only authors actually found): {ag_f}/{n_f} = {100*ag_f/n_f:.1f}%')


## Pipeline metrics against the manual review

We take the manual annotations as **ground truth** and measure how much the automatic pipeline agrees with them. For each dimension we define what is "positive":

| Dimension | Auto-positive means | Manual-positive means |
|---|---|---|
| **author_found** | `author_status='found'` OR `oa_status='found'` | `found_in_ss_manual='yes'` OR `found_in_oa_manual='yes'` |
| **affiliation_match** | `affiliation_status='affiliation_match'` | `affiliation_correct_manual='yes'` |
| **field_match** | `field_status='field_match'` | `field_correct_manual='yes'` |

Metrics reported per dimension:

- **Accuracy** = (TP + TN) / N — total percentage correct (positives and negatives).
- **Precision** = TP / (TP + FP) — of the rows where auto said "positive", how many were really positive.
- **Recall** = TP / (TP + FN) — of the actually positive rows, how many auto detected.
- **F1** = harmonic mean of precision and recall (balance between the two).

For each dimension we print:
1. The **confusion matrix** 2×2 (auto × manual).
2. The 4 metrics in absolute terms (numerator/denominator) and percentage.

Quick interpretation:
- *High precision, low recall* → auto is **cautious**: when it says "positive" it usually gets it right, but it misses cases.
- *Low precision, high recall* → auto is **liberal**: it marks many positives but includes false positives.
- *Both high* → the classifier works well.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Pipeline auto metrics vs. manual annotation (ground truth)
# See explanation of each dimension in the previous cell.
# ────────────────────────────────────────────────────────────────────────────
from libs.factuality.manual_validation import has_text, truthy


from libs.metrics.agreement import report_binary_dimension as report


# ── 1) author_found ──
auto_author   = (df_filled["author_status_auto"] == "found") | (df_filled["oa_status_auto"] == "found")
manual_author = df_filled["found_in_ss_manual"].apply(truthy) | df_filled["found_in_oa_manual"].apply(truthy)
mask_author   = has_text(df_filled["found_in_ss_manual"]) | has_text(df_filled["found_in_oa_manual"])
r1 = report("author_found  (does the person exist?)", auto_author, manual_author, mask_author)

# ── 2) affiliation_match ──
auto_aff   = df_filled.get("affiliation_status_auto", pd.Series("", index=df_filled.index)) == "affiliation_match"
manual_aff = df_filled["affiliation_correct_manual"].apply(truthy)
mask_aff   = has_text(df_filled["affiliation_correct_manual"])
r2 = report("affiliation_match  (does the affiliation match?)", auto_aff, manual_aff, mask_aff)

# ── 3) field_match ──
auto_field   = df_filled.get("field_status_auto", pd.Series("", index=df_filled.index)) == "field_match"
manual_field = df_filled["field_correct_manual"].apply(truthy)
mask_field   = has_text(df_filled["field_correct_manual"])
r3 = report("field_match  (do they work in that field?)", auto_field, manual_field, mask_field)

# ── Compact final summary ──
print("═" * 70)
print("SUMMARY")
print("═" * 70)
summary = pd.DataFrame([r for r in (r1, r2, r3) if r is not None])
if len(summary):
    display(summary[["dim", "n", "TP", "FP", "FN", "TN", "accuracy_%", "precision_%", "recall_%", "f1_%"]])


---

## Alternative sample: 10 random recommendations

Direct sampling on `factuality_full.csv` (1 row = 1 recommendation, not grouped by request). Simpler than the 20-requests flow: it doesn't load JSONs or reconstruct personas — it uses what the pipeline already saved. Uses the `summary/factuality_full.csv` loaded above.

Set `RANDOM_STATE_RECS = None` for a different sample each run; with an int it stays reproducible.

In [ ]:
# ── Generate manual_validation_10_random.csv ─────────────────────────────────
from libs.factuality.manual_validation import search_urls

OUT_RANDOM = OUT_DIR / 'manual_validation_10_random.csv'

N_RECS             = 10
RANDOM_STATE_RECS  = 42        # None for a different sample each run

KEEP_COLS = [
    'model','language','role','task','location','k','target','field','subfield','run_id',
    'name','lastname','current_affiliations','areas_of_research_or_work','reason','source',
    'author_status','oa_status','field_status','seniority_status','location_status',
    'affiliation_status','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all',
    'matched_name','researcher_id','match_score',
    'oa_id','oa_display_name','oa_match_score',
    'gt_field','gt_citations',
]
df_annotator2 = pd.read_csv(FACT_FULL_CSV, low_memory=False,
                    usecols=lambda c: c in KEEP_COLS)
print(f'factuality_full rows: {len(df_annotator2):,}')

sample = df_annotator2.sample(n=N_RECS, random_state=RANDOM_STATE_RECS).reset_index(drop=True)

# Rename to *_auto to align with the 20-requests CSV
rename_auto = {
    'author_status':      'author_status_auto',
    'oa_status':          'oa_status_auto',
    'field_status':       'field_status_auto',
    'seniority_status':   'seniority_status_auto',
    'location_status':    'location_status_auto',
    'affiliation_status': 'affiliation_status_auto',
}
sample = sample.rename(columns=rename_auto)

# Search URLs for the human reviewer
_urls = sample.apply(lambda r: pd.Series(search_urls(r['name'], r['lastname'])), axis=1)
sample[['ss_search_url', 'oa_search_url']] = _urls
# Empty manual columns
for c in ('found_in_ss_manual','found_in_oa_manual','affiliation_correct_manual','field_correct_manual','notes_manual'):
    sample[c] = ''

# Final order
ordered = (
    ['model','language','role','task','location','k','target','field','subfield','run_id',
     'name','lastname','current_affiliations','areas_of_research_or_work','reason','source',
     'author_status_auto','oa_status_auto','field_status_auto','seniority_status_auto','location_status_auto',
     'affiliation_status_auto','affiliation_best_match_score','affiliation_best_match_oa','affiliation_oa_all',
     'matched_name','researcher_id','match_score','gt_field','gt_citations',
     'oa_id','oa_display_name','oa_match_score',
     'ss_search_url','oa_search_url',
     'found_in_ss_manual','found_in_oa_manual','affiliation_correct_manual','field_correct_manual','notes_manual']
)
sample = sample[[c for c in ordered if c in sample.columns]]

sample.to_csv(OUT_RANDOM, index=False)
print(f'Wrote {len(sample)} random recommendations → {OUT_RANDOM}')
sample[['name','lastname','model','language','field','author_status_auto','oa_status_auto','field_status_auto','affiliation_status_auto']]
